In [1]:
import os
import json

kaggle_credentials = {
    "username": "ehtasham9876",
    "key": "KGAT_cc4a2dc421468dc8ead21b4399635eec"
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(kaggle_credentials, f)

os.chmod('/root/.kaggle/kaggle.json', 600)
print("Done!")

Done!


In [2]:
!pip install kaggle -q
!kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:00<00:00, 120MB/s]



In [3]:
!unzip -q cifake-real-and-ai-generated-synthetic-images.zip -d cifake_dataset

In [4]:
import os

for root, dirs, files in os.walk('cifake_dataset'):
    level = root.replace('cifake_dataset', '').count(os.sep)
    if level < 4:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(os.listdir(root))} items)')

cifake_dataset/ (2 items)
  train/ (2 items)
    FAKE/ (50000 items)
    REAL/ (50000 items)
  test/ (2 items)
    FAKE/ (10000 items)
    REAL/ (10000 items)


In [5]:
from PIL import Image
import os

# Check image size
sample_fake = 'cifake_dataset/train/FAKE/' + os.listdir('cifake_dataset/train/FAKE')[0]
sample_real = 'cifake_dataset/train/REAL/' + os.listdir('cifake_dataset/train/REAL')[0]

img = Image.open(sample_fake)
print("Fake image size:", img.size)
print("Fake image mode:", img.mode)

img2 = Image.open(sample_real)
print("Real image size:", img2.size)

Fake image size: (32, 32)
Fake image mode: RGB
Real image size: (32, 32)


CIFAKE → trains in 20 minutes, less impressive results.
ArtiFact → trains in 1-2 hours, much more impressive results , so we will go for  the ArtiFact , lets go

In [6]:
!kaggle datasets download -d awsaf49/artifact-dataset

Dataset URL: https://www.kaggle.com/datasets/awsaf49/artifact-dataset
License(s): other
100% 29.4G/29.4G [05:08<00:00, 102MB/s]



In [7]:
!unzip -q artifact-dataset.zip -d artifact_dataset

In [8]:
import os

for root, dirs, files in os.walk('artifact_dataset'):
    level = root.replace('artifact_dataset', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(os.listdir(root))} items)')

artifact_dataset/ (33 items)
  landscape/ (2 items)
    landscape/ (1 items)
  cycle_gan/ (2 items)
    st/ (3 items)
  pro_gan/ (2 items)
    pro/ (20 items)
  gau_gan/ (2 items)
    gau/ (2 items)
  cips/ (2 items)
    cips/ (3 items)
  lsun/ (6 items)
    cat/ (1 items)
    bedroom/ (1 items)
    car/ (1 items)
    horse/ (1 items)
    church/ (1 items)
  big_gan/ (2 items)
    big/ (1 items)
  stylegan2/ (11 items)
    horse-part2/ (1 items)
    church-part1/ (1 items)
    cat-part1/ (1 items)
    ffhq-part2/ (1 items)
    ffhq-part1/ (1 items)
    church-part2/ (1 items)
    horse-part1/ (1 items)
    car-part1/ (1 items)
    cat-part2/ (1 items)
    car-part2/ (1 items)
  gansformer/ (2 items)
    gf/ (2 items)
  mat/ (2 items)
    mat/ (2 items)
  metfaces/ (2 items)
    metfaces/ (1 items)
  latent_diffusion/ (3 items)
    latentdiff/ (1000 items)
    latentdiff-t2i/ (1 items)
  ffhq/ (2 items)
    ffhq/ (1 items)
  ddpm/ (2 items)
    ddpm/ (2 items)
  afhq/ (2 items)
    afhq

In [9]:
import os

# Look deeper into a few folders to find actual images
test_folders = [
    'artifact_dataset/ffhq/ffhq',
    'artifact_dataset/stylegan2/ffhq-part1',
    'artifact_dataset/coco/coco',
    'artifact_dataset/imagenet/imagenet',
]

for folder in test_folders:
    if os.path.exists(folder):
        contents = os.listdir(folder)
        print(f'{folder}/')
        print(f'  Contents: {contents[:5]}')
        print()

artifact_dataset/ffhq/ffhq/
  Contents: ['images']

artifact_dataset/stylegan2/ffhq-part1/
  Contents: ['ffhq-1024x1024']

artifact_dataset/coco/coco/
  Contents: ['coco2017']

artifact_dataset/imagenet/imagenet/
  Contents: ['val', 'train']



In [10]:
import os

test_folders = [
    'artifact_dataset/ffhq/ffhq/images',
    'artifact_dataset/stylegan2/ffhq-part1/ffhq-1024x1024',
    'artifact_dataset/coco/coco/coco2017',
    'artifact_dataset/imagenet/imagenet/train',
]

for folder in test_folders:
    if os.path.exists(folder):
        contents = os.listdir(folder)
        print(f'{folder}/')
        print(f'  Total items: {len(contents)}')
        print(f'  First 5: {contents[:5]}')
        print()

artifact_dataset/ffhq/ffhq/images/
  Total items: 70000
  First 5: ['img046353.jpg', 'img068504.jpg', 'img005160.jpg', 'img061581.jpg', 'img054158.jpg']

artifact_dataset/stylegan2/ffhq-part1/ffhq-1024x1024/
  Total items: 1
  First 5: ['stylegan2-config-f-psi-0.5']

artifact_dataset/coco/coco/coco2017/
  Total items: 3
  First 5: ['train2017', 'val2017', 'test2017']

artifact_dataset/imagenet/imagenet/train/
  Total items: 1000
  First 5: ['n04579145', 'n04418357', 'n03837869', 'n03337140', 'n02708093']



In [11]:
import os
import shutil
from pathlib import Path

# Create clean real/fake structure
os.makedirs('clean_dataset/train/real', exist_ok=True)
os.makedirs('clean_dataset/train/fake', exist_ok=True)
os.makedirs('clean_dataset/val/real', exist_ok=True)
os.makedirs('clean_dataset/val/fake', exist_ok=True)

# --- REAL IMAGES from FFHQ (70k available, take 25k) ---
real_source = 'artifact_dataset/ffhq/ffhq/images'
real_images = os.listdir(real_source)[:25000]

train_split = int(len(real_images) * 0.8)
for i, fname in enumerate(real_images):
    src = os.path.join(real_source, fname)
    if i < train_split:
        shutil.copy(src, f'clean_dataset/train/real/{fname}')
    else:
        shutil.copy(src, f'clean_dataset/val/real/{fname}')

print(f"Real images copied: {len(real_images)}")

# --- FAKE IMAGES from multiple GAN sources ---
fake_sources = [
    'artifact_dataset/stylegan2/ffhq-part1/ffhq-1024x1024/stylegan2-config-f-psi-0.5',
    'artifact_dataset/stylegan1/sg1',
    'artifact_dataset/pro_gan/pro',
]

fake_images_collected = []
for source in fake_sources:
    if os.path.exists(source):
        for root, dirs, files in os.walk(source):
            for f in files:
                if f.endswith(('.jpg', '.png', '.jpeg')):
                    fake_images_collected.append(os.path.join(root, f))

print(f"Fake images found: {len(fake_images_collected)}")

Real images copied: 25000
Fake images found: 150000


In [12]:
import random
random.seed(42)

# Take only 25000 fake images to match real count
fake_images_selected = random.sample(fake_images_collected, 25000)

train_split = int(25000 * 0.8)  # 20000 train, 5000 val

for i, src in enumerate(fake_images_selected):
    fname = f'fake_{i:05d}.jpg'
    if i < train_split:
        shutil.copy(src, f'clean_dataset/train/fake/{fname}')
    else:
        shutil.copy(src, f'clean_dataset/val/fake/{fname}')

print("Fake images copied!")

# Final count
print("\nDataset Summary:")
print(f"Train real: {len(os.listdir('clean_dataset/train/real'))}")
print(f"Train fake: {len(os.listdir('clean_dataset/train/fake'))}")
print(f"Val real:   {len(os.listdir('clean_dataset/val/real'))}")
print(f"Val fake:   {len(os.listdir('clean_dataset/val/fake'))}")

Fake images copied!

Dataset Summary:
Train real: 20000
Train fake: 20000
Val real:   5000
Val fake:   5000


In [13]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision import models
from torch.utils.data import DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = ImageFolder(root='clean_dataset/train', transform=train_transform)
val_dataset   = ImageFolder(root='clean_dataset/val',   transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)

print("Classes:", train_dataset.classes)
print("Train batches:", len(train_loader))
print("Val batches:",   len(val_loader))

Classes: ['fake', 'real']
Train batches: 1250
Val batches: 313


In [14]:
# Build model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

picture_model = models.efficientnet_b0(weights='IMAGENET1K_V1')

for param in picture_model.parameters():
    param.requires_grad = False

picture_model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(1280, 256),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(256, 1),
    nn.Sigmoid()
)

picture_model = picture_model.to(device)
print("Device:", device)
print("Model ready!")

# Training setup
criterion = nn.BCELoss()
optimizer = optim.Adam(picture_model.classifier.parameters(), lr=0.001)
scheduler = StepLR(optimizer, step_size=3, gamma=0.5)

# Train
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        predicted = (outputs >= 0.5).float()
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        if (batch_idx + 1) % 100 == 0:
            print(f'  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f}')
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            predicted = (outputs >= 0.5).float()
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

# Run 5 epochs
EPOCHS = 5
for epoch in range(EPOCHS):
    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print('-' * 40)
    train_loss, train_acc = train_epoch(picture_model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(picture_model, val_loader, criterion, device)
    scheduler.step()
    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}%')
    print(f'Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc*100:.2f}%')

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 66.8MB/s]


Device: cuda
Model ready!

Epoch 1/5
----------------------------------------
  Batch 100/1250 | Loss: 0.4548
  Batch 200/1250 | Loss: 0.2568
  Batch 300/1250 | Loss: 0.3048
  Batch 400/1250 | Loss: 0.2212
  Batch 500/1250 | Loss: 0.2893
  Batch 600/1250 | Loss: 0.2662
  Batch 700/1250 | Loss: 0.2414
  Batch 800/1250 | Loss: 0.4820
  Batch 900/1250 | Loss: 0.3427
  Batch 1000/1250 | Loss: 0.3083
  Batch 1100/1250 | Loss: 0.1339
  Batch 1200/1250 | Loss: 0.2455
Train Loss: 0.3254 | Train Acc: 86.28%
Val Loss:   0.2483 | Val Acc:   90.45%

Epoch 2/5
----------------------------------------
  Batch 100/1250 | Loss: 0.3723
  Batch 200/1250 | Loss: 0.2910
  Batch 300/1250 | Loss: 0.1944
  Batch 400/1250 | Loss: 0.3446
  Batch 500/1250 | Loss: 0.1347
  Batch 600/1250 | Loss: 0.1872
  Batch 700/1250 | Loss: 0.3152
  Batch 800/1250 | Loss: 0.6197
  Batch 900/1250 | Loss: 0.3855
  Batch 1000/1250 | Loss: 0.2796
  Batch 1100/1250 | Loss: 0.2372
  Batch 1200/1250 | Loss: 0.3594
Train Loss: 0.2797

In [15]:
torch.save(picture_model.state_dict(), 'picture_model.pth', _use_new_zipfile_serialization=False)
print("Picture model saved!")

from google.colab import files
files.download('picture_model.pth')

Picture model saved!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>